<a href="https://colab.research.google.com/drive/187PwBsrodSKbdU-7j5L5AVfw832tXOZ8?usp=sharing" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# SAS-Prompt for Robot Self-Improvement

> Use feedback from simulated golf shots to improve robot actions through self-adaptive prompting.

| At a glance | |
|---|---|
| **Level** | Advanced |
| **Estimated runtime** | 30–45 minutes |
| **Category** | Reinforcement learning and optimization |
| **Required service** | Google Gemini API |
| **Recommended environment** | Google Colab |

## Notebook map

1. [Overview and concepts](#overview)
2. [Set up the notebook](#setup)
3. [Run the demo](#run-demo)
4. [Evaluate and explore](#evaluate)
5. [Summary and next steps](#summary)

> **Run order:** Execute cells from top to bottom. Add your key as `GEMINI_API_KEY` in Colab Secrets; if it is unavailable, the setup cell will prompt for it securely. API calls may incur usage charges depending on your account.


<a id="overview"></a>

## 1. Overview and concepts


This notebook provides a step-by-step walkthrough of an interactive demonstration showing how Large Language Models (LLMs) can act as numerical optimizers, an adaptation of the *[SAS Prompt](https://arxiv.org/pdf/2504.20459)* (Ben Amor et al., 2025) framework introduced for robot self-improvement. In the original paper, the authors show that **LLMs can autonomously self-improve a robot policy** by repeatedly performing an (a) **S**ummarization step, (b) and **A**nalysis step and (c) a **S**ynthesis step, hence the abbreviation "SAS". The result is a policy optimization process that is similar to reinforcement learning but which is built upon the LLMs reasoning capabilities.

Our demo replaces table-tennis robot in the original paper (Ben Amor et al., 2025) with a simplified golf-robot simulation. In our demo, the robot attempts to reach a target hole by adjusting its launch parameters (a = azimuth, b = velocity, c = elevation). After each shot, the LLM receives execution outcomes and iteratively proposes improved parameters based on past trials and objective feedback. **The goal is to demonstrate how an LLM—without being told the physics equations—can treat robot performance traces as data, retrieve examples relevant to the goal, analyze parameter effects, and synthesize improved actions**. This mirrors the paper’s key insight: LLMs can perform stochastic numerical optimization entirely inside a prompt, forming a textual analogue of a gradient that drives self-improvement over multiple iterations.

### Overview
This demonstration guides you through:
- Simulating 5 golf-ball trajectories for different (a, b, c) launch parameters and passing them in as the initial examples for ICL
- Recording each attempt’s landing (x, y)
- Packaging previous shots as in-context examples for the SAS-style prompt
- Iteratively refining the shot accuracy through an LLM-driven optimization loop

The result is a clear visualization of how LLMs behave as gradient-free optimizers, closely following the Summarize → Analyze → Synthesize structure demonstrated in the paper.

### Background

This demo adapts key concepts from the SAS-Prompt methodology presented in the original work(Ben Amor et al., 2025):
1. LLMs as Numerical Optimizers: the SAS-Prompt paper shows that LLMs can perform global optimization on functions such as Ackley and Rastrigin—using no derivative information—by proposing successive candidate values and adjusting based on observed outcomes. This emergent ability enables gradient-free policy search directly inside the LLM.

2. Summarize → Analyze → Synthesize (SAS): In the original table-tennis domain, the LLM receives robot execution traces (positions, paddle trajectories, landing points) and learns to: Summarize all prior examples into a structured table, Analyze which parameters influence the objective, and Synthesize new attenuation parameters to improve performance. This loop allows the LLM to gradually discover correlations—for example, that increasing certain parameters (e.g., “g” or “h” in the paper) shifts landing locations rightward.

3. Our Adaptation to a Golf Robot: Instead of analyzing 8D attenuation vectors and 3D ball trajectories, the LLM receives: launch parameters (a, b, c), a simulated landing coordinate (x_hit, y_hit). The LLM must infer how azimuth, velocity, and elevation shape projectile motion and propose better parameters—mirroring the self-improvement loop in SAS.

**Why This Is Interesting**

Just as the original paper demonstrates robot behavior optimization without explicit reward functions or gradient updates, this demo shows that the LLM infers parameter–behavior relationships (a textual gradient) and in turn uses that to optimize entirely through a combination of language and numbers. Throughout our example there is no reward function. The objective is specified in language terms. In addition, there is no external training or any other external component: all learning and adaptation occurs within the LLM. A critical aspect of this demo is the repeated iterative calls to the LLM. In each episode, the LLM proposes a new set of parameters which are run in the Golf simulation. Instead of a reward value, a table containing information about the execution is returned back to the LLM. In turn, by analyzing this table and previous executions, the LLM proposes a new, better set of parameters to try. This is performed in a loop. Hence, this setup benchmarks the LLM’s ability to conduct explainable, in-context, and iterative optimization based solely on observed results.

### Let’s Take a Look at an Example

This first example shows five randomly generated trajectories that are being sent to the LLM as the initial examples, these examples contain the variables for azimuth, velocity, trajectory, and the x,y coordinate of the landing spot of the ball.

<img src="https://raw.githubusercontent.com/intro-to-icl/intro-to-icl.github.io/refs/heads/master/static/images/golf_trajectories.png" height="500">

Then the LLM proposes new parameters (a, b, c) and receives execution outcomes, it repeats this process iteratively, proposing improved parameters based on past trials and objective feedback. This continues after the optimization loop ends and displays to following visualization, where the golf robot hits the ball, and lands at a spot based on the last predicted (a, b, c) parameters.

<br>
<img src="https://raw.githubusercontent.com/intro-to-icl/intro-to-icl.github.io/refs/heads/master/static/images/golf_robot.gif" height="500">

### LLM as the Optimizer

As in the original SAS prompt experiments, we use Gemini as the LLM, but this time with versions 2.0/2.5 Flash.

The workflow follows the SAS structure:

1. Summarize: The LLM organizes past shots into a table, extracting features such as: parameter values (a, b, c), and the resulting (x_hit, y_hit)

2. Analyze: The model reasons about correlations, forming a textual gradient: i.e. “Increasing b tends to overshoot the target.”, “Reducing a by 1–2 degrees shifts the landing left.”, or “Higher elevation c produces shorter carry distances.” This mirrors the parameter-effect explanations found in the table-tennis domain analysis steps .

3. Synthesize: A new parameter triple (a, b, c) is proposed that improves upon all previous trials—analogous to synthesizing new attenuation vectors in SAS. This update is then executed, logged, and appended to the in-context examples for the next iteration. Through repeated application, the LLM performs explainable, gradient-free optimization, gradually steering the robot toward the target.

### Code Overview
The implementation is organized into a modular structure, with each component responsible for a different stage of the SAS Prompt pipeline. This design separates data generation, visualization, prompt construction, LLM inference, and result interpretation, making the system easy to understand, modify, and extend.


<a id="setup"></a>

## 2. Set up the notebook

Run the setup cells below once before continuing.


In [ ]:
#@title **Import Necessary Libraries**
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google import genai
from google.genai import types
import re
import itertools
import math
import ast
import json
import ipywidgets as widgets
from IPython.display import display
import os
import getpass
import matplotlib.animation as animation
from IPython.display import HTML
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


In [ ]:
#@title **Connect to Gemini securely**
import getpass

try:
    from google.colab import userdata
    apikey = userdata.get("GEMINI_API_KEY")
except Exception:
    apikey = None

if not apikey:
    apikey = getpass.getpass("Enter your Gemini API key: ")

if not apikey:
    raise ValueError("A Gemini API key is required to run this demo.")
client = genai.Client(api_key=apikey)



In [ ]:
#@title **Choose a Model**
model_dropdown = widgets.Dropdown(
    options=[
        ("Gemini 2.5 Flash", "gemini-2.5-flash"),
        ("Gemini 2.0 Flash", "gemini-2.0-flash")
    ],
    description="Model:",
    value="gemini-2.5-flash",
    style={'description_width': 'initial'}
)

confirm_button = widgets.Button(
    description="Confirm Selection"
)

output = widgets.Output()

model_name = None

def on_confirm_click(b):
    global model_name, batch_size

    model_name = model_dropdown.value

    with output:
        output.clear_output()
        print(f"\nSelected model: {model_name}")

confirm_button.on_click(on_confirm_click)

display(model_dropdown, confirm_button, output)

### Utility Functions

The cell `Golf Physics Simulation Functions` is in charge of simulating the path of a golf shot. It utlizes physics functions and a gravity variable to calculate the x,y coordinate of the ball's landing position.

The cell `Initial Target & Examples Generation` initialaizes a target hole and five random example golf shots, then simulates each shot. For each shot, it randomly picks an azimuth angle between –10° and 10°, a launch speed between 40 and 70 m/s, and an elevation angle between 5° and 25°. Each shot gets stimulated using the physics function and outputs a list with five simulated golf shots, each with both trajectory data and the parameters that created it.

Then, the cell `Initial Target & Examples Plot` plots each of the five example shots so that the user can easily see what the examples being passed into the LLM look like, providing a visualization that's easy to understand.


In [ ]:
#@title **Golf Physics Simulation Functions**
def simulate_shot(a, b, c, wind=(0, 0), g=9.81, n_points=100):
    az = np.radians(a)   # azimuth angle
    el = np.radians(c)   # elevation angle

    # Initial velocities
    v0x = b * np.cos(el) * np.cos(az)
    v0y = b * np.cos(el) * np.sin(az)
    v0z = b * np.sin(el)

    # Basic projectile time of flight
    t_flight = 2 * v0z / g if v0z > 0 else 0

    # Time samples
    t = np.linspace(0, t_flight, n_points)

    # Wind as a velocity offset (basic physics)
    vx = v0x + wind[0]
    vy = v0y + wind[1]

    # Trajectory
    x = vx * t
    y = vy * t
    z = v0z * t - 0.5 * g * t**2

    # Clip below ground
    z[z < 0] = 0

    return {
        "x": x,
        "y": y,
        "z": z,
        "x_hit": x[-1],
        "y_hit": y[-1],
        "t_flight": t_flight
    }

In [ ]:
#@title **Initial Target & Examples Generation**
np.random.seed(0)
x_target, y_target = 300, 5   # hole location
target = np.array([x_target, y_target])

N = 5
a = np.random.uniform(-10, 10, N)     # azimuth degrees
b = np.random.uniform(40, 70, N)      # velocity m/s
c = np.random.uniform(5, 25, N)       # elevation degrees

shots = []
for i in range(N):
    sim = simulate_shot(a[i], b[i], c[i])
    sim.update({"a": a[i], "b": b[i], "c": c[i]})
    shots.append(sim)


In [ ]:
#@title **Initial Target & Examples Plot**
from mpl_toolkits.mplot3d import Axes3D

def plot_golf_shots(shots, target):
    fig = plt.figure(figsize=(9, 9))
    ax = fig.add_subplot(111, projection='3d')

    for i, s in enumerate(shots):
        ax.plot(s["x"], s["y"], s["z"], lw=2, label=f"Shot {i+1}")
        ax.scatter(s["x_hit"], s["y_hit"], 0, marker="o", s=40)
        ax.text(s["x_hit"], s["y_hit"], 0, fontsize=8, s="")

    # Target hole marker
    ax.scatter(target[0], target[1], 0, c='r', s=120, marker='*', label="Hole")

    # Labels and styling
    ax.set_xlabel("X distance (m)")
    ax.set_ylabel("Y distance (m)")
    ax.set_ylim(-5,20)
    ax.set_zlim(0,50)
    ax.set_zlabel("Height (m)")
    ax.set_title("3D Golf Shot Trajectory Examples")
    ax.legend()
    ax.grid(True)
    ax.view_init(elev=25, azim=-60)
    plt.show()

plot_golf_shots(shots, target)

<a id="run-demo"></a>

## 3. Run the demo

Follow the cells in order and read each note before making an API call.


The cell below `SAS Prompt Building` builds the text prompt given to the LLM during each iteration. It formats the five random shot examples, the target hole and the history of previously proposed parameter values with their landing coordinates, then inserts them into a structured instruction block. The function returns a prompt that tells the model to propose better parameter values to hit the target.

### Let's take a look at the prompt

```
Domain Description:
A robotic golf club launches a ball with initial velocity magnitude b, elevation angle c, and azimuth a (direction).
The ball follows a 3D parabolic trajectory under gravity = 9.81 m/s².

Objective:
Land as close as possible to the target hole at position ({target[0]}, {target[1]}) in meters.

Examples:
{initial_shots and previous_predicted_shots}

Step 1: Create a table using the examples, noting which parameter sets land closer to the target based on their hit coordinates.
Step 2: From the table, give me the parameters that are closest to fulfilling the objective.
Step 3: Take these parameters and the summary table and analyze the directional trends of the control parameters. Let's think step by step!
Step 4: Determine how increasing or decreasing each of a, b, and c shifts the landing point in x and y
Step 5: Based on this analysis, propose a new set of [a, b, c] parameter values which will bring us closer to the objective than any of the previous examples.
Step 6: Return **ONLY** the final parameters in this exact format:
```

This prompt gives the LLM all the information it needs to act as an optimizer for the getting the correct parameters for the golf shot. It begins with a short description of the physical setup: a robotic golf club launches a ball using three controllable parameters—azimuth a, velocity b, and elevation c—and the ball follows a normal projectile path under gravity. After defining the physics context, the prompt states the objective, which is to land the ball as close as possible to the actual target hole. It then lists all the example shots produced so far, including both the initial random ones and the LLM-generated attempts, each labeled with their parameters and landing locations.

Following the examples, the prompt gives the model a structured sequence of steps intended to guide its reasoning. These steps tell the LLM to evaluate which past shots performed well, understand how changes in each parameter affect the landing point, and then propose a new set of parameters that should move the shot closer to the target. The prompt also instructs the LLM to be exploratory when needed and avoid repeating old values. Finally, it restricts the output to only the parameter triple a= , b= , c= with no extra explanation, ensuring clean results for the next optimization iteration and eventual final visualizations.


In [ ]:
#@title **SAS Prompt Building**
def build_sas_prompt(examples, target):
    table_rows = "\n".join([
        f"{i+1}. a={ex['a']:.2f}, b={ex['b']:.2f}, c={ex['c']:.2f}, hit(x,y)=({ex['x_hit']:.1f},{ex['y_hit']:.1f})"
        for i, ex in enumerate(examples)
    ])
    return f"""
Domain Description:
A robotic golf club launches a ball with direction a, initial velocity magnitude b, elevation angle c.
The ball follows a 3D parabolic trajectory under gravity = 9.81 m/s².

Objective:
Land as close as possible to the target hole at position ({target[0]}, {target[1]}) in meters.

Examples:
{table_rows}

Step 1: Create a table using the examples, noting which parameter sets land closer to the target based on their hit coordinates.
Step 2: From the table, give me the parameters that are closest to fulfilling the objective.
Step 3: Take these parameters and the summary table and analyze the directional trends of the control parameters. Let's think step by step!
Step 4: Determine how increasing or decreasing each of a, b, and c shifts the landing point in x and y
Step 5: Based on this analysis, propose a new set of [a, b, c] parameter values which will bring us closer to the objective than any of the previous examples.
Step 6: Return **ONLY** the final parameters in this exact format:
a= , b= , c=
"""


The cell below `Optimization Loop (a = azimuth, b = velocity, c = elevation)` runs a full iterative process where the LLM repeatedly proposes new golf-shot parameters. It begins each iteration by simulating all previously generated shots using their stored parameter values and updating their landing points.

It then constructs a prompt containing the updated shots and the target location, and sends this prompt to the LLM, asking it to propose improved values of a, b, and c. After receiving the LLM’s response, the loop extracts the proposed parameter values using a regular expression; if the extraction succeeds, it converts them to floats, simulates a new shot using those proposed parameters, and appends that new simulated shot to the growing list of examples.

This repeats for a fixed number of iterations (25), gradually building a sequence of LLM-generated golf shots intended to move closer to the target.


In [ ]:
#@title **Optimization Loop**

n_iterations = 25
for iteration in range(n_iterations):
    print("------------------------------------------------------------------------------------------------")
    print(f"Iteration {iteration+1}")

    prompt = build_sas_prompt(shots, target)

    try:
        response = client.models.generate_content(
            model=model_name,
            contents=[prompt]
        )
        llm_reply = response.text
        print("Proposed Parameters:", llm_reply)

    except Exception as e:
        print(f"Error in Iteration {iteration}")
        print("Pausing for 60 seconds before continuing...")
        time.sleep(60)
        continue

    match = re.findall(
        r"a\s*=\s*([-+]?\d*\.?\d+).*?b\s*=\s*([-+]?\d*\.?\d+).*?c\s*=\s*([-+]?\d*\.?\d+)",
        llm_reply,
        flags=re.DOTALL
    )

    if match:
        a_str, b_str, c_str = [re.sub(r"[^\d\.\-]", "", val) for val in match[-1]]

        try:
            a_new, b_new, c_new = map(float, (a_str, b_str, c_str))
        except ValueError:
            print("Parsing error: could not convert parameters to float, skipping iteration.")
            continue

        # Simulate new shot (produces x_hit, y_hit)
        sim_new = simulate_shot(a_new, b_new, c_new)
        sim_new.update({"a": a_new, "b": b_new, "c": c_new})
        # Ensure no error key is added
        sim_new.pop("error", None)

        shots.append(sim_new)

        print(
            f"Added new shot: a={a_new:.3f}, b={b_new:.3f}, c={c_new:.3f}, "
            f"x_hit={sim_new['x_hit']:.3f}, y_hit={sim_new['y_hit']:.3f}"
        )

    else:
        print("Could not find parameter pattern from Gemini output, continuing to next iteration")
print ("Completed")

### LLM Example Response to SAS-Prompt (raw output)
This is the exact response format taken from one iteration of the optimization loop. It follows the exact steps we gave in the SAS-Prompt, and it shows how the model clearly lays out each part of the analysis—from listing the examples, to comparing distances, to explaining the trends it sees, and finally giving the new parameters in the required format. The output is organized, easy to read, and mirrors the structure we asked for, making it straightforward to understand how the model arrived at its final suggestion.

---
### Raw Output

Iteration 3:

Here's a step-by-step analysis and proposal:

#### Step 1: Create a table using the examples, noting which parameter sets land closer to the target based on their hit coordinates.

Target Hole: (300, 5)

| Example | a      | b      | c      | hit(x) | hit(y) | Delta X (hit-target) | Delta Y (hit-target) | Distance to Target |
| :------ | :----- | :----- | :----- | :----- | :----- | :------------------- | :------------------- | :----------------- |
| 1       | 0.98   | 59.38  | 20.83  | 238.9  | 4.1    | -61.1                | -0.9                 | 61.106             |
| 2       | 4.30   | 53.13  | 15.58  | 148.4  | 11.2   | -151.6               | 6.2                  | 151.727            |
| 3       | 2.06   | 66.75  | 16.36  | 245.4  | 8.8    | -54.6                | 3.8                  | 54.732             |
| 4       | 0.90   | 68.91  | 23.51  | 354.1  | 5.5    | 54.1                 | 0.5                  | 54.102             |
| 5       | -1.53  | 51.50  | 6.42   | 60.1   | -1.6   | -239.9               | -6.6                 | 239.991            |
| 6       | 0.93   | 67.83  | 19.94  | 300.7  | 4.9    | 0.7                  | -0.1                 | 0.707              |
| 7       | 0.96   | 67.80  | 19.94  | 300.4  | 5.0    | 0.4                  | 0.0                  | 0.400              |

#### Step 2: From the table, give me the parameters that are closest to fulfilling the objective.

The parameters closest to the objective are from **Example 7**, with a distance of 0.400:
a = 0.96, b = 67.80, c = 19.94
Hit coordinates: (300.4, 5.0)

#### Step 3: Take these parameters and the summary table and analyze the directional trends of the control parameters.

Let's focus on the two closest examples: Example 7 (best) and Example 6 (second best), as they represent minor adjustments around the target.

*   **Example 7:** a=0.96, b=67.80, c=19.94, hit=(300.4, 5.0)
*   **Example 6:** a=0.93, b=67.83, c=19.94, hit=(300.7, 4.9)

Comparing Example 7 to Example 6:
*   Δa = 0.93 - 0.96 = -0.03
*   Δb = 67.83 - 67.80 = +0.03
*   Δc = 19.94 - 19.94 = 0.00
*   Resulting Δx = 300.7 - 300.4 = +0.3
*   Resulting Δy = 4.9 - 5.0 = -0.1

General trends observed from the table and physical intuition:
*   **Parameter 'a' (Direction):**
    *   Larger 'a' values tend to increase the y-coordinate of the landing point. (e.g., Ex5 a=-1.53, y=-1.6; Ex2 a=4.30, y=11.2. Also Ex7 a=0.96, y=5.0 vs Ex6 a=0.93, y=4.9 confirms: decreasing 'a' decreases 'y').
*   **Parameter 'b' (Initial Velocity Magnitude):**
    *   Larger 'b' values tend to increase the x-coordinate (range) of the landing point. (e.g., Ex1 b=59.38, x=238.9 vs Ex4 b=68.91, x=354.1).
*   **Parameter 'c' (Elevation Angle):**
    *   For angles below 45 degrees (which all examples are), larger 'c' values tend to increase the x-coordinate (range) of the landing point. (e.g., Ex1 c=20.83, x=238.9; Ex4 c=23.51, x=354.1 - though 'b' also increased here).

#### Step 4: Determine how increasing or decreasing each of a, b, and c shifts the landing point in x and y.

Based on the analysis, especially from comparing Ex7 and Ex6, and general physics:

*   **Increasing 'a'**: Shifts the landing point primarily in the **positive y direction**. (Decreasing 'a' shifts in negative y).
    *   *Sensitivity from Ex7 to Ex6:* Δy / Δa ≈ -0.1 / -0.03 ≈ +3.33 units of y per unit of a (assuming minimal cross-effect).
*   **Increasing 'b'**: Shifts the landing point primarily in the **positive x direction** (increasing range). (Decreasing 'b' shifts in negative x).
    *   *Sensitivity from Ex7 to Ex6:* Δx / Δb ≈ +0.3 / +0.03 ≈ +10 units of x per unit of b (assuming minimal cross-effect).
*   **Increasing 'c'**: Shifts the landing point primarily in the **positive x direction** (increasing range, for angles < 45 degrees). (Decreasing 'c' shifts in negative x).
    *   No clean empirical data for 'c' sensitivity, but theoretical models suggest range is proportional to sin(2c).

#### Step 5: Based on this analysis, propose a new set of [a, b, c] parameter values which will bring us closer to the objective than any of the previous examples.

Current best hit (Ex 7): (300.4, 5.0) with parameters a=0.96, b=67.80, c=19.94.
Target: (300.0, 5.0)

We need to adjust the hit point from (300.4, 5.0) to (300.0, 5.0).
This means we need:
*   Δx = -0.4 (reduce x by 0.4 meters)
*   Δy = 0.0 (keep y at 5.0 meters)

Applying the trends and sensitivities:

1.  **Adjust 'y'**: Current y is already 5.0, and the target y is 5.0. To maintain this, and assuming 'a' primarily controls 'y' with minimal cross-effects from 'b' or 'c', we should keep 'a' constant.
    *   Proposed `a` = 0.96

2.  **Adjust 'x'**: Current x is 300.4, target x is 300.0. We need to decrease 'x' by 0.4 meters. We can do this by decreasing 'b' or 'c'.
    Using the sensitivity of 'x' to 'b' derived from Ex7/Ex6: Δx / Δb = +10.
    To achieve Δx = -0.4, we need:
    Δb = Δx / 10 = -0.4 / 10 = -0.04.
    *   Proposed `b` = 67.80 - 0.04 = 67.76.

3.  **Adjust 'c'**: Since we are primarily using 'b' to adjust 'x', and 'c' was constant in the Ex6/Ex7 comparison, we will keep 'c' constant as well to simplify the adjustment.
    *   Proposed `c` = 19.94.

This new set of parameters (a=0.96, b=67.76, c=19.94) makes a targeted adjustment based on the most relevant empirical data to reduce the x-coordinate while maintaining the y-coordinate.

#### Step 6: Return ONLY the final parameters in this exact format:

a=0.96, b=67.76, c=19.94

---


<a id="evaluate"></a>

## 4. Evaluate and explore

Inspect the outputs, compare methods where available, and try changing one input at a time.


This cell below plots the distance error for each proposed shot across iterations. It visualizes a graph where the error is plotted, and displays the decrease in error in the LLMs predictions over iteraitons.


In [ ]:
#@title **Error Over Iterations**
df = pd.DataFrame(shots)
df["error"] = np.sqrt((df["x_hit"] - x_target)**2 + (df["y_hit"] - y_target)**2)
plt.figure(figsize=(12,5))

plt.plot(df['error'], color = 'red', marker='o')
plt.title('Error Over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Error')
plt.grid(True)
plt.show()

This cell below visualizes the final predicted trajectory using the proposed (a,b,c) parameters from the last iteration. It starts off witha zoomed-in view of the robot hitting the ball, and after pressing play, zooms out and displays the trajectory of the ball and it's landing spot.


In [ ]:
#@title **Plot Gemini-Predicted Trajectory (press ▶ to play)**
def set_line_data_3d(line, x, y, z):
    line.set_data(x, y)
    line.set_3d_properties(z)

def robot_parts(ax, swing_angle_deg=-170, base_y=0):
    """Draw robot body (no head), base, and arm."""
    body_line, = ax.plot([0, 0], [base_y, base_y], [0, 1.0], c="black", lw=4)
    base_line, = ax.plot([-0.1, 0.1], [base_y-0.1, base_y+0.1], [0, 0], c="black", lw=6)
    arm_len = 0.6
    angle = np.radians(swing_angle_deg)
    x_arm = [0, arm_len * np.cos(angle)]
    y_arm = [base_y, base_y + 0.1]
    z_arm = [0.8, 0.8 + arm_len * np.sin(angle)]
    arm_line, = ax.plot(x_arm, y_arm, z_arm, c="orange", lw=4, zorder=4)
    return body_line, base_line, arm_line

final_row = df.iloc[-1]
x_vals = np.array(final_row["x"])
y_vals = np.array(final_row["y"])
z_vals = np.array(final_row["z"])
x_hit = final_row["x_hit"]
y_hit = final_row["y_hit"]

x_target, y_target = 300, 5   # hole location
target = np.array([x_target, y_target])

x0, y0, z0 = x_vals[0], y_vals[0], z_vals[0]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.view_init(elev=25, azim=-60)
ax.set_title("Gemini-Predicted Trajectory (Golf Robot Simulation)")

ax.xaxis.pane.set_facecolor((0.8, 0.9, 1.0, 0.0))
ax.yaxis.pane.set_facecolor((0.8, 0.9, 1.0, 0.0))
ax.zaxis.pane.set_facecolor((0.2, 0.6, 0.2, 0.9))

ax.set_xlim(x0 - 1, x0 + 3)
ax.set_ylim(y0 - 1, y0 + 1)
ax.set_zlim(0, 2)
ax.set_xlabel("X distance (m)")
ax.set_ylabel("Y distance (m)")
ax.set_zlabel("Height (m)")

ax.scatter(target[0], target[1], 0, c="dodgerblue", s=140, marker="*", label="Hole")

line_path, = ax.plot([], [], [], lw=2, color="red", label="Shot Path")

ball = ax.scatter(x0, y0, z0, c="white", s=40, edgecolor="black", zorder=6)
body_line, base_line, arm_line = robot_parts(ax, swing_angle_deg=-90, base_y=y0)
ax.legend(loc="upper right")

MAX_FRAMES = len(x_vals)
impact_frame = 0
zoom_frames = 60

XMAX = max(350, x_hit + 50)
YRANGE = 40
ZMAX = 80

def init():
    set_line_data_3d(line_path, [], [], [])
    ball._offsets3d = ([x0], [y0], [z0])
    return [line_path, arm_line, ball]

def update(frame):
    idx = min(frame, MAX_FRAMES - 1)

    if frame < impact_frame:
        set_line_data_3d(line_path, [], [], [])
    else:
        set_line_data_3d(line_path, x_vals[:idx+1], y_vals[:idx+1], z_vals[:idx+1])

    swing_angle = np.clip(-170 + (frame * 3), -120, -30)
    arm_len = 0.6
    x_arm = [0, arm_len * np.cos(np.radians(swing_angle))]
    y_arm = [y0, y0 + 0.1]
    z_arm = [0.8, 0.8 + arm_len * np.sin(np.radians(swing_angle))]
    set_line_data_3d(arm_line, x_arm, y_arm, z_arm)

    xb, yb, zb = x_vals[idx], y_vals[idx], z_vals[idx]

    if frame < impact_frame:
        ball._offsets3d = ([x0], [y0], [z0])
    elif frame < impact_frame + 5:
        f = (frame - impact_frame) / 5
        f = np.clip(f, 0, 1)
        x_smooth = x0 + f * (x_vals[5] - x0)
        y_smooth = y0 + f * (y_vals[5] - y0)
        z_smooth = z0 + f * (z_vals[5] - z0)
        ball._offsets3d = ([x_smooth], [y_smooth], [z_smooth])
    else:
        ball._offsets3d = ([xb], [yb], [zb])

    if frame < impact_frame + zoom_frames:
        f = (frame - impact_frame) / zoom_frames
        f = np.clip(f, 0, 1)
        ax.set_xlim(x0 - 1 + f * (0 - (x0 - 1)), x0 + 3 + f * (XMAX - (x0 + 3)))
        ax.set_ylim(y0 - 1 + f * (-30 - (y0 - 1)), y0 + 1 + f * (40 - (y0 + 1)))
        ax.set_zlim(0, 2 + f * (ZMAX - 2))
    else:
        ax.set_xlim(0, XMAX)
        ax.set_ylim(-30, 40)
        ax.set_zlim(0, ZMAX)

    return [line_path, arm_line, ball]

ani = FuncAnimation(fig, update, frames=MAX_FRAMES, init_func=init,
                    interval=40, blit=False, repeat=False)

ani.save("golf_robot.gif", writer="pillow", fps=25)
plt.close(fig)
HTML(ani.to_jshtml())

### Extended Demo: Alternative Objective

This alternative Objective is almost identical in structure to the original SAS prompt, but it changes the optimization goal. In the original version, the LLM is instructed to choose parameters that make the ball land as close as possible to the actual target hole, whose coordinates are given. In contrast, the alternative version tells the LLM to aim for a point located 60m to the right of the target hole. Everything else—the domain description, the example formatting, the seven-step reasoning scaffold, and the requirement to output only “a= , b= , c=”—remains the same. The only meaningful difference is the objective text, which shifts the task from direct target-hitting to landing at the midpoint distance.

```
Domain Description:
A robotic golf club launches a ball with initial velocity magnitude b, elevation angle c, and azimuth a (direction).
The ball follows a 3D parabolic trajectory under gravity = 9.81 m/s².

**KEY DIFFERENCE**
Objective:
Your goal is to choose parameters [a, b, c] that make the ball **land exactly 60 meters to the right** of the target hole
The target hole is located at ({target[0]}, {target[1]}).
******************

Examples:
{initial_shots and previous_predicted_shots}

Step 1: Create a table using the examples, noting which parameter sets land closer to the target based on their hit coordinates.
Step 2: From the table, give me the parameters that are closest to fulfilling the objective.
Step 3: Take these parameters and the summary table and analyze the directional trends of the control parameters. Let's think step by step!
Step 4: Based on these directional trends, determine in which direction each parameter must move relative to ALL prior examples, not just the closest ones. Avoid using interpolation between nearby examples. Use global reasoning.
Step 5: Based on this analysis, propose a new set of [a, b, c] parameter values which will bring us closer to the objective than any of the previous examples.
Step 6: Return **ONLY** the final parameters in this exact format:
a= , b= , c=
```


In [ ]:
#@title **Alternative Objective SAS Prompt Building Function (hitting 60 meters to the right of the target hole)**
def build_alt_sas_prompt(examples, target):
    table_rows = "\n".join([
        f"{i+1}. a={ex['a']:.2f}, b={ex['b']:.2f}, c={ex['c']:.2f}, hit(x,y)=({ex['x_hit']:.1f},{ex['y_hit']:.1f})"
        for i, ex in enumerate(examples)
    ])
    return f"""
Domain Description:
A robotic golf club launches a ball with direction a, initial velocity magnitude b, elevation angle c.
The ball follows a 3D parabolic trajectory under gravity = 9.81 m/s².

Objective:
Your goal is to choose parameters [a, b, c] that make the ball **land exactly 60 meters to the right** of the target hole
The target hole is located at ({target[0]}, {target[1]}).

Examples:
{table_rows}

Step 1: Create a table using the examples, noting which parameter sets land closer to the target based on their hit coordinates.
Step 2: From the table, give me the parameters that are closest to fulfilling the objective.
Step 3: Take these parameters and the summary table and analyze the directional trends of the control parameters. Let's think step by step!
Step 4: Based on these directional trends, determine in which direction each parameter must move relative to ALL prior examples, not just the closest ones. Avoid using interpolation between nearby examples. Use global reasoning.
Step 5: Based on this analysis, propose a new set of [a, b, c] parameter values which will bring us closer to the objective than any of the previous examples.
Step 6: Return **ONLY** the final parameters in this exact format:
a= , b= , c=
"""


In [ ]:
#@title **Alternative Objective Optimization Loop**
np.random.seed(0)
x_target, y_target = 300, 5   # hole location
target = np.array([x_target, y_target])

N = 5
a = np.random.uniform(-10, 10, N)     # azimuth degrees
b = np.random.uniform(40, 70, N)      # velocity m/s
c = np.random.uniform(5, 25, N)       # elevation degrees

alt_shots = []
for i in range(N):
    sim = simulate_shot(a[i], b[i], c[i])
    sim.update({"a": a[i], "b": b[i], "c": c[i]})
    alt_shots.append(sim)

for s in alt_shots:
    s["error"] = np.sqrt((s["x_hit"] - x_target)**2 + (s["y_hit"] - y_target)**2)

n_iterations = 50
for iteration in range(n_iterations):
    print(f"Iteration {iteration+1}")

    prompt = build_alt_sas_prompt(alt_shots, target)
    try:
        response = client.models.generate_content(
            model=model_name,
            contents=[prompt]
        )
        llm_reply = response.text
        print("Proposed Parameters:", llm_reply)
        print("-----------------------------------------------------------")

    except Exception as e:
        print(f"Error in Iteration {iteration}")
        print("Pausing for 60 seconds before continuing...")
        time.sleep(60)
        continue

    match = re.findall(
        r"a\s*=\s*([-+]?\d*\.?\d+).*?b\s*=\s*([-+]?\d*\.?\d+).*?c\s*=\s*([-+]?\d*\.?\d+)",
        llm_reply,
        flags=re.DOTALL
    )

    if match:
        a_str, b_str, c_str = [re.sub(r"[^\d\.\-]", "", val) for val in match[-1]]

        try:
            a_new, b_new, c_new = map(float, (a_str, b_str, c_str))
        except ValueError:
            print("Parsing error: could not convert parameters to float, skipping iteration.")
            continue

        # Simulate new shot (produces x_hit, y_hit)
        sim_new = simulate_shot(a_new, b_new, c_new)
        sim_new.update({"a": a_new, "b": b_new, "c": c_new})
        # Ensure no error key is added
        sim_new.pop("error", None)

        alt_shots.append(sim_new)

        print(
            f"Added new shot: a={a_new:.3f}, b={b_new:.3f}, c={c_new:.3f}, "
            f"x_hit={sim_new['x_hit']:.3f}, y_hit={sim_new['y_hit']:.3f}"
        )

    else:
        print("Could not find parameter pattern from Gemini output, continuing to next iteration")


### LLM Example Response to SAS-Prompt w/ Alternative Objective
This is the exact response format taken from one iteration of the optimization loop. By using the alternative objective **(hit to the right of the target by 60 meters)**, it infers the new target it needs to hit the ball to and generates the best parameter values to do so.

Similar to the previous example, it follows the exact steps we gave in the SAS-Prompt, and it shows how the model clearly lays out each part of the analysis—from listing the examples, to comparing distances, to explaining the trends it sees, and finally giving the new parameters in the required format. The output is organized, easy to read, and mirrors the structure we asked for, making it straightforward to understand how the model arrived at its final suggestion.

---

### Raw Output

Iteration 7:

Proposed Parameters: Okay, let's break this down step-by-step.

**Step 1: Create a table using the examples, noting which parameter sets land closer to the target based on their hit coordinates.**

The target hole is at (300, 5).  We want the ball to land at (300 + 60, 5) = (360, 5). I'll calculate the distance of each hit from this target and sort by distance.

| Example | a    | b     | c     | hit(x, y) | Distance from (360, 5) |
|---------|------|-------|-------|-----------|-----------------------|
| 5       | -1.53| 51.50 | 6.42  | (60.1,-1.6)| 305.5                 |
| 2       | 4.30 | 53.13 | 15.58 | (148.4,11.2)| 211.8                 |
| 1       | 0.98 | 59.38 | 20.83 | (238.9,4.1) | 121.1                 |
| 3       | 2.06 | 66.75 | 16.36 | (245.4,8.8) | 114.8                 |
| 7       | 0.88 | 68.50 | 23.00 | (344.0,5.3) | 16.0                  |
| 4       | 0.90 | 68.91 | 23.51 | (354.1,5.5) | 5.9                   |
| 9       | 0.92 | 69.00 | 23.70 | (357.2,5.7) | 2.8                   |
| 11      | 0.92 | 69.50 | 23.70 | (362.4,5.8) | 2.4                   |
| 6       | 0.85 | 70.00 | 24.50 | (376.9,5.6) | 16.9                  |
| 10      | 0.92 | 69.70 | 24.10 | (369.1,5.9) | 9.1                   |
| 8       | 1.05 | 70.50 | 24.80 | (385.8,7.1) | 25.9                  |

**Step 2: From the table, give me the parameters that are closest to fulfilling the objective.**

Example 11 is the closest, with hit(x,y) = (362.4, 5.8) and a distance of 2.4 from the target (360, 5).

**Step 3: Take these parameters and the summary table and analyze the directional trends of the control parameters. Let's think step by step!**

*   **'a' (Direction):** It's difficult to see a clear trend for 'a' based on the given data because it ranges from negative values to values over 4. However, the best values seem to cluster near 0.9. The positive values indicate movement to the right of the initial heading (0).
*   **'b' (Initial Velocity):** Higher velocities are generally required to get closer to the target. However, the highest velocity value of all examples sends the ball too far.
*   **'c' (Elevation Angle):** The elevation angle seems to correlate with distance, with low values being too short and higher values potentially overshooting.

**Step 4: Based on these directional trends, determine in which direction each parameter must move relative to ALL prior examples, not just the closest ones. Avoid using interpolation between nearby examples. Use global reasoning.**

*   **'a' (Direction):** Example 11's 'a' value is 0.92. To get *closer* to 360, we might slightly decrease 'a' to reduce the rightward trajectory, but it's already very close. However, the value cannot be any higher than the highest example and we've identified an appropriate range for this parameter. Decreasing is most likely to succeed.

*   **'b' (Initial Velocity):** Example 11's 'b' value is 69.5. The x-coordinate is slightly higher than the target, so to move *closer* to the target, we can slightly decrease velocity.

*   **'c' (Elevation Angle):** Example 11's 'c' value is 23.7. The y-coordinate is slightly above 5, so it needs to come down a bit. Slightly *decrease* the angle to shorten the trajectory.

**Step 5: Based on this analysis, propose a new set of [a, b, c] parameter values which will bring us closer to the objective than any of the previous examples.**

Based on the above trends, I'll slightly decrease 'a', 'b', and 'c' from Example 11's values:

a = 0.90, b = 69.3, c = 23.6

**Step 6: Return ONLY the final parameters in this exact format:**

a=0.90, b=69.3, c=23.6


In [ ]:
#@title **Plot Error Over Iterations for Alternative Objective (3D Golf Simulation)**
alt_df = pd.DataFrame(alt_shots)
x_target_new = x_target+60
y_target_new = y_target
alt_df["error"] = np.sqrt((alt_df["x_hit"] - x_target_new)**2 + (alt_df["y_hit"] - y_target_new)**2)
plt.figure(figsize=(12,5))

plt.plot(alt_df['error'], color = 'red', marker='o')
plt.title('Error Over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Error')
plt.grid(True)
plt.show()

In [ ]:
#@title **Plot Gemini-Predicted Trajectory with Alternative Objective (press ▶ to play)**
def set_line_data_3d(line, x, y, z):
    line.set_data(x, y)
    line.set_3d_properties(z)

def robot_parts(ax, swing_angle_deg=-170, base_y=0):
    """Draw robot body (no head), base, and arm."""
    body_line, = ax.plot([0, 0], [base_y, base_y], [0, 1.0], c="black", lw=4)
    base_line, = ax.plot([-0.1, 0.1], [base_y-0.1, base_y+0.1], [0, 0], c="black", lw=6)
    arm_len = 0.6
    angle = np.radians(swing_angle_deg)
    x_arm = [0, arm_len * np.cos(angle)]
    y_arm = [base_y, base_y + 0.1]
    z_arm = [0.8, 0.8 + arm_len * np.sin(angle)]
    arm_line, = ax.plot(x_arm, y_arm, z_arm, c="orange", lw=4, zorder=4)
    return body_line, base_line, arm_line

final_row = alt_df.iloc[-1]
x_vals = np.array(final_row["x"])
y_vals = np.array(final_row["y"])
z_vals = np.array(final_row["z"])
x_hit = final_row["x_hit"]
y_hit = final_row["y_hit"]

x_target, y_target = 300, 5   # distance between origin and original hole location
x_target_2, y_target_2 = x_target+60, y_target
target = np.array([x_target, y_target])
target_2 = np.array([x_target_2, y_target_2])

x0, y0, z0 = x_vals[0], y_vals[0], z_vals[0]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
ax.view_init(elev=25, azim=-60)
ax.set_title("Gemini-Predicted Trajectory (Golf Robot Simulation)")

ax.xaxis.pane.set_facecolor((0.8, 0.9, 1.0, 0.0))
ax.yaxis.pane.set_facecolor((0.8, 0.9, 1.0, 0.0))
ax.zaxis.pane.set_facecolor((0.2, 0.6, 0.2, 0.9))

ax.set_xlim(x0 - 1, x0 + 3)
ax.set_ylim(y0 - 1, y0 + 1)
ax.set_zlim(0, 2)
ax.set_xlabel("X distance (m)")
ax.set_ylabel("Y distance (m)")
ax.set_zlabel("Height (m)")

ax.scatter(target[0], target[1], 0, c="purple", s=140, marker="*", label="Original Target")
ax.scatter(target_2[0], target_2[1], 0, c="dodgerblue", s=140, marker="*", label="New Point")

line_path, = ax.plot([], [], [], lw=2, color="red", label="Shot Path")

ball = ax.scatter(x0, y0, z0, c="white", s=40, edgecolor="black", zorder=6)
body_line, base_line, arm_line = robot_parts(ax, swing_angle_deg=-90, base_y=y0)
ax.legend(loc="upper right")

MAX_FRAMES = len(x_vals)
impact_frame = 0
zoom_frames = 60

XMAX = max(350, x_hit + 50)
YRANGE = 40
ZMAX = 80

def init():
    set_line_data_3d(line_path, [], [], [])
    ball._offsets3d = ([x0], [y0], [z0])
    return [line_path, arm_line, ball]

def update(frame):
    idx = min(frame, MAX_FRAMES - 1)

    if frame < impact_frame:
        set_line_data_3d(line_path, [], [], [])
    else:
        set_line_data_3d(line_path, x_vals[:idx+1], y_vals[:idx+1], z_vals[:idx+1])

    swing_angle = np.clip(-170 + (frame * 3), -120, -30)
    arm_len = 0.6
    x_arm = [0, arm_len * np.cos(np.radians(swing_angle))]
    y_arm = [y0, y0 + 0.1]
    z_arm = [0.8, 0.8 + arm_len * np.sin(np.radians(swing_angle))]
    set_line_data_3d(arm_line, x_arm, y_arm, z_arm)

    xb, yb, zb = x_vals[idx], y_vals[idx], z_vals[idx]

    if frame < impact_frame:
        ball._offsets3d = ([x0], [y0], [z0])
    elif frame < impact_frame + 5:
        f = (frame - impact_frame) / 5
        f = np.clip(f, 0, 1)
        x_smooth = x0 + f * (x_vals[5] - x0)
        y_smooth = y0 + f * (y_vals[5] - y0)
        z_smooth = z0 + f * (z_vals[5] - z0)
        ball._offsets3d = ([x_smooth], [y_smooth], [z_smooth])
    else:
        ball._offsets3d = ([xb], [yb], [zb])

    if frame < impact_frame + zoom_frames:
        f = (frame - impact_frame) / zoom_frames
        f = np.clip(f, 0, 1)
        ax.set_xlim(x0 - 1 + f * (0 - (x0 - 1)), x0 + 3 + f * (XMAX - (x0 + 3)))
        ax.set_ylim(y0 - 1 + f * (-30 - (y0 - 1)), y0 + 1 + f * (40 - (y0 + 1)))
        ax.set_zlim(0, 2 + f * (ZMAX - 2))
    else:
        ax.set_xlim(0, XMAX)
        ax.set_ylim(-30, 40)
        ax.set_zlim(0, ZMAX)

    return [line_path, arm_line, ball]

ani = FuncAnimation(fig, update, frames=MAX_FRAMES, init_func=init,
                    interval=40, blit=False, repeat=False)

ani.save("golf_robot_alt.gif", writer="pillow", fps=25)
plt.close(fig)
HTML(ani.to_jshtml())

<a id="summary"></a>

## 5. Summary and next steps


### Summary

This demo illustrates how LLMs, utilizing the SAS Prompt framework, can perform in-context numerical optimization by iteratively improving parameter values using only execution traces and objective feedback contained within the prompt. By presenting a small set of initial (a, b, c) launch parameters along with the history of previously attempted shots and their corresponding landing coordinates, the model is encouraged to infer the underlying parameter–trajectory relationships and propose better values over time—without being told any physics equations. Each iteration provides updated context from real shot outcomes, allowing the LLM to adjust its proposals and gradually steer the robot toward the target through pattern recognition rather than explicit gradient computation. Through this setup, we observe how the model implicitly constructs a “textual gradient,” effectively behaving as an optimizer driven purely by context, consistent with the SAS Prompt framework.

Beyond this golf-robot example, the same approach generalizes to a wide range of robot-learning and numerical reasoning tasks. By embedding execution traces, intermediate outcomes, or constraint violations directly inside the prompt, LLMs can be guided to refine policies, tune continuous control parameters, or adapt behaviors in simulated or real environments. These variations highlight the flexibility of the SAS Prompt framework for enabling self-improvement loops, demonstrating how LLMs can perform iterative optimization when structured through Summarize → Analyze → Synthesize prompts.

### Conclusion

This demonstration shows that LLMs can act as effective gradient-free numerical optimizers using only the information supplied in the prompt—capturing the core idea of the SAS Prompt framework in the setting of a simplified robot-golf simulation. Without receiving reward functions, analytic derivatives, or external optimization rules, the LLM learns from the pattern of outcomes and parameter updates to generate increasingly accurate (a, b, c) values over successive iterations. Although the model is not performing true physical simulation or mathematical optimization internally, its ability to approximate improved launch parameters purely from contextual feedback validates the original paper’s insight: LLMs can drive robot self-improvement through structured prompting alone. Our reimplementation shows that the SAS Prompt approach works well in this domain and effectively proves the concept, demonstrating a practical and explainable way for LLMs to optimize robotic behavior using language as the optimization substrate.

<br>

For more details, please refer to the [SAS-Prompt Project Page](https://sites.google.com/asu.edu/sas-llm/) and the associated [research paper](https://arxiv.org/pdf/2504.20459).

### References
Ben Amor, H., Graesser, L., Iscen, A., D’Ambrosio, D., Abeyruwan, S., Bewley, A., Zhou, Y., Kalirathinam, K., Mishra, S., & Sanketi, P. (2025, May). SAS-prompt: Large Language Models as Numerical Optimizers for Robot Self-Improvement. In *2025 IEEE International Conference on Robotics and Automation (ICRA)* (pp. 10087-10094). IEEE.
